# Lab 4 — Multimodal LLMs [SOLUTION]

**LLMs for NLP | First Year Masters**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiberiu44/master-unibuc-2026/blob/main/labs/LLMs%20for%20NLP%20-%20Lab%204%20-%20Multimodality/solution_lab.ipynb)

---

**what if the input isn't text at all?**

> *"What counts as a token is a choice."*

---

**What you will build:**
1. **Section 0** — patch tokenization vs. word tokenization
2. **Section 1** — `PatchEmbedding` from scratch
3. **Section 2** — bridging vision and language spaces
4. **Section 3** — CLIP Contrastive Loss: a completely different alignment strategy
5. **Section 4** — Fine-Tuning Florence-2 on ChartQA with LoRA
6. **Section 5** — Failure Mode Hunt

In [ ]:
!pip install "transformers>=4.40.0,<5.0.0" "tokenizers>=0.15.0,<0.20.0" "torchao==0.16.0"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForCausalLM, CLIPProcessor, CLIPModel
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader, Dataset
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — Sections 0-3 work fine on CPU. Section 4 will be slow; see fallback note.')

---
## Section 0: The Provocation


---

### 0.1 The Comparison Table

Fill in the `TODO` cells below. You know the text side from Labs 2 and 3.

| Property | Text Token (Lab 2) | Image Patch (Lab 4) |
|---|---|---|
| What is the "atom" being tokenized? | A BPE subword unit | A fixed-size rectangular patch of pixels (e.g., 16×16×3 = 768 values) |
| How is position encoded? | Sinusoidal / learned index | Learned 2D positional embedding (one per patch position, flattened to 1D index) |
| What information is lost? | Subword boundaries (original characters) | Sub-patch texture detail — all spatial structure WITHIN a patch is compressed into one vector |
| Shape after embedding | `(seq_len, d_model)` | `(num_patches + 1, d_model)`; +1 for the CLS token |
| Pre-processing step | BPE tokenization | Normalize pixel values, resize to fixed resolution (e.g., 224×224) |

### 0.2 Visualizing Patch Tokenization

In [ ]:
# Generate a colorful 64x64 test image programmatically
np.random.seed(42)
raw = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
# Add some structure so it looks interesting
for i in range(0, 64, 8):
    raw[i:i+8, :, 0] = np.linspace(50, 200, 64, dtype=np.uint8)
    raw[:, i:i+8, 2] = np.linspace(200, 50, 64, dtype=np.uint8).reshape(-1, 1)

img = Image.fromarray(raw)
PATCH_SIZE = 16

img_array = np.array(img)
h, w = img_array.shape[:2]
patches_per_row = w // PATCH_SIZE
patches_per_col = h // PATCH_SIZE
total_patches = patches_per_row * patches_per_col

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: the original image
axes[0].imshow(img_array)
axes[0].set_title(f'Original image ({w}×{h} pixels)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Right: the image with patch grid overlaid
axes[1].imshow(img_array)
for x in range(0, w + 1, PATCH_SIZE):
    axes[1].axvline(x - 0.5, color='white', linewidth=1.5, alpha=0.8)
for y in range(0, h + 1, PATCH_SIZE):
    axes[1].axhline(y - 0.5, color='white', linewidth=1.5, alpha=0.8)

# Label each patch
for row in range(patches_per_col):
    for col in range(patches_per_row):
        patch_num = row * patches_per_row + col
        cx = col * PATCH_SIZE + PATCH_SIZE // 2
        cy = row * PATCH_SIZE + PATCH_SIZE // 2
        axes[1].text(cx, cy, str(patch_num), color='white', fontsize=7,
                     ha='center', va='center', fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.4))

axes[1].set_title(f'Tokenized: {total_patches} patches (patch size {PATCH_SIZE}×{PATCH_SIZE})',
                  fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.suptitle('"What counts as a token is a choice."\n'
             f'This 64×64 image becomes {total_patches} tokens, each a {PATCH_SIZE}×{PATCH_SIZE}×3 chunk of pixels.',
             fontsize=11, style='italic', y=1.01)
plt.tight_layout()
plt.show()

print(f"Image: {w}x{h} pixels")
print(f"Patch size: {PATCH_SIZE}x{PATCH_SIZE}")
print(f"Total patches (tokens): {total_patches}")
print(f"Each patch has {PATCH_SIZE * PATCH_SIZE * 3} raw values (H*W*RGB channels)")
print(f"Compare: a typical BERT sentence ~ 30 tokens")

---
## Section 1: Image Tokenization — PatchEmbedding


### 1.1 Annotate the PatchEmbedding



In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        # SOLUTION: Computes the total number of non-overlapping patches.
        # For a 224x224 image with patch_size=16: (224/16)^2 = 14^2 = 196 patches.
        # We square because patches tile the image in both height and width.

        self.projection = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        # SOLUTION: This Conv2d does TWO things simultaneously:
        #   1. EXTRACTS each non-overlapping patch (kernel_size=patch_size handles the window size)
        #   2. PROJECTS each patch from (patch_size*patch_size*in_channels) values to d_model values
        # stride=patch_size ensures no overlap: the filter jumps by exactly one patch width each time.
        # This is equivalent to: reshape patch → linear layer, but faster on GPU.

        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model))
        # SOLUTION: We add 1 because we prepend a CLS token to the sequence.
        # The positional embedding must cover num_patches image positions PLUS the CLS position.
        # It's a learnable Parameter (not a fixed sinusoid) — ViT learns the best positional encoding.

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        # SOLUTION: The [CLS] token is a special learned vector prepended before all patch tokens.
        # After attention, its output vector is used as the global image representation.
        # You saw this in BERT (Lab 3): the [CLS] output was used for classification.
        # Here it serves the same purpose — a "summary" of the entire image.

    def forward(self, x):
        # x: (B, C, H, W)
        B = x.shape[0]

        x = self.projection(x)       # shape: (B, d_model, H/P, W/P)
        # SOLUTION: The Conv2d transforms (B, C, H, W) → (B, d_model, H/patch_size, W/patch_size).
        # For 224x224 with patch=16: (B, 768, 14, 14).
        # The spatial dims (14, 14) tell us there are 14x14=196 patches arranged in a grid.

        x = x.flatten(2)             # shape: (B, d_model, num_patches)
        # SOLUTION: We flatten the spatial dimensions (dim 2 and 3) into one sequence dimension.
        # (B, 768, 14, 14) → (B, 768, 196).
        # This converts the 2D patch grid into a 1D sequence, which is what the Transformer expects.

        x = x.transpose(1, 2)        # shape: (B, num_patches, d_model)
        # SOLUTION: Transformer convention is (B, sequence_length, features).
        # After flatten we have (B, d_model, num_patches) — d_model is in dim 1, sequence in dim 2.
        # Transpose swaps them to (B, num_patches, d_model) — sequence first, features second.

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embedding
        return x

In [ ]:
pe = PatchEmbedding(img_size=224, patch_size=16, d_model=768)
test_img = torch.randn(2, 3, 224, 224)
out = pe(test_img)
assert out.shape == (2, 197, 768), f"Expected (2, 197, 768), got {out.shape}"
print(f"✅ PatchEmbedding: 224x224 image → {out.shape[1]} tokens × {out.shape[2]}-dim each")
print(f"   (196 patches + 1 CLS token = 197 total)")
print(f"   Compare to BERT: 512 text tokens max — images are EXPENSIVE in attention!")

### 1.2 Visualizing What Each Patch "Sees"

In [ ]:
# Visualize a 224x224 image split into 16x16 patches — show the first 16 patches
np.random.seed(7)
raw_224 = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
# Add structure: horizontal gradient in red, vertical in blue
for r in range(224):
    raw_224[r, :, 0] = int(r / 224 * 200 + 20)
for c in range(224):
    raw_224[:, c, 2] = int(c / 224 * 200 + 20)

img224 = Image.fromarray(raw_224)
img_arr = np.array(img224)

P = 16  # patch size
patches_per_row = 224 // P  # = 14

fig = plt.figure(figsize=(16, 5))

# Original image with grid
ax_main = fig.add_subplot(1, 3, 1)
ax_main.imshow(img_arr)
for x in range(0, 225, P):
    ax_main.axvline(x - 0.5, color='white', linewidth=0.5, alpha=0.6)
for y in range(0, 225, P):
    ax_main.axhline(y - 0.5, color='white', linewidth=0.5, alpha=0.6)
ax_main.set_title(f'224×224 → {14*14} patch tokens\n(+ 1 CLS = 197 total)', fontsize=10, fontweight='bold')
ax_main.axis('off')

# Show first 16 patches individually
ax_patches = fig.add_subplot(1, 3, 2)
patch_grid = np.zeros((4 * P, 4 * P, 3), dtype=np.uint8)
for i in range(16):
    row_i = i // 4
    col_i = i % 4
    orig_row = i // patches_per_row
    orig_col = i % patches_per_row
    patch = img_arr[orig_row*P:(orig_row+1)*P, orig_col*P:(orig_col+1)*P]
    patch_grid[row_i*P:(row_i+1)*P, col_i*P:(col_i+1)*P] = patch

ax_patches.imshow(patch_grid)
for x in range(0, 4*P+1, P):
    ax_patches.axvline(x - 0.5, color='white', linewidth=1)
for y in range(0, 4*P+1, P):
    ax_patches.axhline(y - 0.5, color='white', linewidth=1)
for i in range(16):
    r, c = i // 4, i % 4
    ax_patches.text(c*P + P//2, r*P + P//2, str(i), color='white',
                    fontsize=9, ha='center', va='center', fontweight='bold')
ax_patches.set_title('First 16 patches\n(each becomes 1 token)', fontsize=10, fontweight='bold')
ax_patches.axis('off')

# Show the projection concept
ax_proj = fig.add_subplot(1, 3, 3)
ax_proj.axis('off')
ax_proj.set_xlim(0, 10)
ax_proj.set_ylim(0, 10)

ax_proj.add_patch(plt.Rectangle((0.5, 6), 3, 3, fill=True, facecolor='#4C72B0', alpha=0.7, edgecolor='black'))
ax_proj.text(2, 7.5, f'Patch\n{P}×{P}×3\n={P*P*3} values', ha='center', va='center',
             fontsize=9, color='white', fontweight='bold')

ax_proj.annotate('', xy=(6, 7.5), xytext=(4, 7.5),
                 arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax_proj.text(5, 8.1, 'Conv2d\n(linear proj)', ha='center', va='center', fontsize=8, color='#333')

ax_proj.add_patch(plt.Rectangle((6.5, 5.5), 2.5, 4, fill=True, facecolor='#DD8452', alpha=0.7, edgecolor='black'))
ax_proj.text(7.75, 7.5, 'Token\n768-dim\nvector', ha='center', va='center',
             fontsize=9, color='white', fontweight='bold')

ax_proj.text(5, 4.5, 'The Conv2d IS the patch tokenizer\nAND the embedding projector.\nOne operation, two jobs.', 
             ha='center', va='center', fontsize=8.5, style='italic',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax_proj.set_title('Patch → Token embedding', fontsize=10, fontweight='bold')

plt.suptitle('Vision Transformer Tokenization: PatchEmbedding internals', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Questions [SOLUTION]

**1.** A 224×224 image with patch size 16 produces how many patch tokens? **196**

   Attention is O(n²). Text has ~30 tokens -> 900 pairs. Images have 197 tokens → 38,809 pairs.
   Image self-attention is ~43x more expensive per example than sentence self-attention.
   This is why efficient attention variants (Flash Attention, window attention) matter for vision models.

---

**2.** `stride=patch_size` means the convolution kernel jumps by exactly `patch_size` pixels between applications.
   With `stride=16` and `kernel_size=16`, each kernel application covers pixels [0:16], [16:32], [32:48]... No overlap.

   With `stride=1`, the kernel would slide 1 pixel at a time, producing overlapping patches and O(H*W) tokens instead of O((H/P)*(W/P)).

---

**3.** Within each 16×16 patch (768 values), all pixel-level spatial structure is compressed into a single 768-dim vector.
   You cannot tell where within the patch a particular edge or texture was. That sub-patch spatial information is gone.
   BPE loses different information (exact character spelling) but is similar in spirit: you trade fine-grained detail for compact sequence length.

In [ ]:
class ProjectionMLP(nn.Module):
    """
    Maps vision embeddings into the language model's token embedding space.
    This is the 'translator' between two independently pre-trained models.

    Architecture: Linear → GELU → Linear
    Input:  (B, N, d_vision)   — N patch tokens from the vision encoder
    Output: (B, N, d_llm)      — N tokens ready for the language model
    """
    def __init__(self, d_vision=768, d_llm=512, hidden_dim=1024):
        super().__init__()
        self.fc1 = nn.Linear(d_vision, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, d_llm)

    def forward(self, x):
        # x shape: (B, N, d_vision)
        # SOLUTION: 2-layer MLP with GELU activation.
        # PyTorch Linear works on any leading batch dimensions,
        # so (B, N, d_vision) → (B, N, hidden_dim) → (B, N, d_llm) automatically.
        x = self.fc1(x)   # (B, N, d_vision) → (B, N, hidden_dim)
        x = self.act(x)   # GELU non-linearity
        x = self.fc2(x)   # (B, N, hidden_dim) → (B, N, d_llm)
        return x

In [ ]:
proj = ProjectionMLP(d_vision=768, d_llm=512, hidden_dim=1024)
test_vision_tokens = torch.randn(2, 197, 768)
projected = proj(test_vision_tokens)
assert projected is not None, "❌ forward() returned None — did you forget to return?"
assert projected.shape == (2, 197, 512), f"❌ Expected (2, 197, 512), got {projected.shape}"
print(f"✅ ProjectionMLP: (2, 197, 768) → {projected.shape}")
print(f"   Vision space (d_vision=768) mapped to LLM space (d_llm=512)")

### 2.2 The Garbage Output Demo

In [ ]:
patch_embed = PatchEmbedding().eval()
proj_untrained = ProjectionMLP().eval()  # RANDOMLY INITIALIZED — not trained

with torch.no_grad():
    fake_image = torch.randn(1, 3, 224, 224)
    vision_tokens = patch_embed(fake_image)
    projected = proj_untrained(vision_tokens)

print("Vision tokens (raw ViT output):")
print(f"  Shape: {vision_tokens.shape}")
print(f"  Mean: {vision_tokens.mean():.4f}, Std: {vision_tokens.std():.4f}")
print()
print("After UNTRAINED projection:")
print(f"  Shape: {projected.shape}")
print(f"  Mean: {projected.mean():.4f}, Std: {projected.std():.4f}")
print()
print("Warning: These outputs are MEANINGLESS — the projection is randomly initialized.")
print("   The language model would see incoherent 'visual tokens'.")
print("   Fine-tuning teaches the projection what these tokens should mean.")

### 📝 Diagnostic Questions [SOLUTION]

**1.** The loss still computes because the forward pass is fully differentiable regardless of weight initialization.
   The randomly initialized projection produces numbers (not NaNs) — they're just in the wrong region of the LLM's embedding space.
   The LLM then generates tokens based on those garbage inputs. The generation is random-looking, but the computation graph is intact,
   so cross-entropy loss compares the LLM output distribution against the correct target and gradients flow back through everything.

---

**2.** The vision encoder was trained on billions of image-text pairs (e.g., CLIP trained on 400M pairs).
   The language model was trained on trillions of text tokens.
   Both have learned highly structured, useful representations.
   Training them from scratch on 150 examples would massively overfit — the models would memorize those 150 charts
   and generalize to nothing. The projection is much smaller (a 2-layer MLP) and only needs to learn the alignment, not vision or language.

---

**3.** Inject-once (LLaVA): visual tokens enter at the input layer only. The language model processes them like text tokens.
   Simpler, faster, but the visual signal can fade as it passes through many layers (the "forgetting" problem).
   Inject repeatedly (Flamingo): cross-attention at every decoder layer means the model can "look back" at the image at each generation step.
   More expressive, handles long documents where visual reference is needed far from the injection point — but costs 2x compute per decoder layer.

### 3.2 The Typographic Attack

In [ ]:
print("Loading CLIP (openai/clip-vit-base-patch32)...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("✅ CLIP loaded")

def clip_classify(image: Image.Image, categories: list) -> dict:
    """Zero-shot classify an image into one of the given text categories."""
    inputs = clip_processor(
        text=categories,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=-1)[0]
    return {cat: f"{p:.1%}" for cat, p in zip(categories, probs)}

def add_text_overlay(image: Image.Image, text: str) -> Image.Image:
    """Add large text overlay to an image (typographic attack)."""
    img = image.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    try:
        font = ImageFont.load_default(size=48)
    except TypeError:
        font = ImageFont.load_default()
    draw.text((w // 6, h // 2 - 24), text, fill=(255, 255, 255), font=font)
    return img

# Load a real dog image from CIFAR-10 (label 5 = dog)
print("Loading a real dog image from CIFAR-10...")
cifar = load_dataset("cifar10", split="test")
dog_sample = cifar.filter(lambda x: x["label"] == 5)[0]
dog_clean = dog_sample["img"].convert("RGB").resize((224, 224))
dog_attacked = add_text_overlay(dog_clean, "iPod")
print("✅ Dog image loaded")

categories = ["a dog", "an iPod", "a cat", "a car"]

print("\nClean dog image:")
clean_result = clip_classify(dog_clean, categories)
print(clean_result)

print("\nSame image + 'iPod' text overlaid:")
attacked_result = clip_classify(dog_attacked, categories)
print(attacked_result)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(dog_clean)
axes[0].set_title(f"Clean image\n" + str(clean_result), fontsize=9)
axes[0].axis('off')
axes[1].imshow(dog_attacked)
axes[1].set_title(f"Typographic attack\n" + str(attacked_result), fontsize=9)
axes[1].axis('off')
plt.suptitle('The Typographic Attack: Writing "iPod" on a dog image fools CLIP', fontsize=11)
plt.tight_layout()
plt.show()

print()
print("This is the Typographic Attack (Goh et al., 2021).")
print("CLIP's dual-encoder reads text IN the image and uses it for classification.")
print("It has no explicit separation between 'text in image' and 'semantic image content'.")
print()
print("This happens because CLIP was trained to align WHOLE embeddings.")
print("It has no mechanism to say: 'ignore the text that happens to be in this photo'.")

---
## Section 4: Fine-Tuning a Real Vision-Language Model

In [ ]:
MODEL_ID = "multimodalart/Florence-2-large-no-flash-attn"
TASK_PREFIX = "<DocVQA>"  # Try changing to "<OCR>" or "<CAPTION>" later!

print("Loading Florence-2-base processor and model...")
print("(This may take a moment on first run — weights are ~850MB)")
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=torch.float32)
    model = model.to(device)
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✅ Florence-2-base loaded ({total_params:.0f}M parameters)")
    FLORENCE_AVAILABLE = True
except Exception as e:
    print(f"❌ Failed to load Florence-2: {e}")
    print("   If you are on CPU with limited RAM, try: MODEL_ID = 'microsoft/Florence-2-base'")
    print("   and ensure trust_remote_code=True is set.")
    FLORENCE_AVAILABLE = False

### 4.1 Zero-Shot Baseline

In [ ]:
import io as _io

def open_image(img):
    """Handle PIL Image or raw bytes returned by HuggingFace datasets."""
    if isinstance(img, bytes):
        return Image.open(_io.BytesIO(img))
    return img

print("Loading ChartQA dataset...")
chartqa = load_dataset("ahmed-masry/chartqa", split="test").select(range(100))
eval_examples = [chartqa[i] for i in range(5)]
print(f"✅ Loaded {len(chartqa)} test examples")

def run_inference(mdl, proc, image, question, task_prefix=TASK_PREFIX):
    """Run Florence-2 inference on a single image+question pair."""
    inputs = proc(
        images=image,
        text=task_prefix + question,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated_ids = mdl.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=64,
            num_beams=3
        )

    raw = proc.batch_decode(generated_ids, skip_special_tokens=False)[0]
    try:
        parsed = proc.post_process_generation(
            raw, task=task_prefix,
            image_size=(image.width, image.height)
        )
        return parsed.get(task_prefix, str(parsed))
    except Exception:
        return raw

print("\n── Zero-Shot Baseline (before fine-tuning) ──────────────────────\n")
zero_shot_results = []
for i, ex in enumerate(eval_examples):
    image = open_image(ex["image"]).convert("RGB")
    question = ex["query"]
    ground_truth = ex["label"][0] if isinstance(ex["label"], list) else ex["label"]

    prediction = run_inference(model, processor, image, question)
    zero_shot_results.append({"question": question, "gt": ground_truth, "pred": prediction})

    print(f"Example {i+1}:")
    print(f"  Q: {question}")
    print(f"  GT: {ground_truth}")
    print(f"  Pred (zero-shot): {prediction}")
    print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(3):
    axes[i].imshow(open_image(eval_examples[i]["image"]))
    gt = eval_examples[i]["label"][0] if isinstance(eval_examples[i]["label"], list) else eval_examples[i]["label"]
    q_short = eval_examples[i]["query"][:40]
    axes[i].set_title(f"Q: {q_short}...\nGT: {gt}", fontsize=8)
    axes[i].axis('off')
plt.suptitle("Zero-Shot Baseline Examples (before fine-tuning)", fontsize=11)
plt.tight_layout()
plt.show()

### 4.2 Data Pipeline Inspection

The most important thing to understand before training: **what does the processor actually return?**

In [ ]:
sample = chartqa[0]
sample_image = open_image(sample["image"]).convert("RGB")
sample_question = TASK_PREFIX + sample["query"]

encoding = processor(
    images=sample_image,
    text=sample_question,
    return_tensors="pt"
)

print("── What the processor returns ──────────────────────────────────")
for key, value in encoding.items():
    print(f"  {key}: shape={value.shape}, dtype={value.dtype}")

print(f"\n── Token breakdown ─────────────────────────────────────────────")
input_ids = encoding["input_ids"][0]
tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
print(f"  Total tokens: {len(tokens)}")
print(f"  First 20 tokens: {tokens[:20]}")
print(f"  Decoded: {processor.tokenizer.decode(input_ids)[:120]}...")

print(f"\n── Image tensor breakdown ──────────────────────────────────────")
pv = encoding["pixel_values"][0]
print(f"  pixel_values shape: {pv.shape}  (channels × height × width)")
print(f"  Value range: [{pv.min():.3f}, {pv.max():.3f}]")
estimated_patches = (pv.shape[1] // 16) * (pv.shape[2] // 16)
print(f"  Estimated patch tokens (with patch_size=16): {estimated_patches}")

print(f"\n── Processing the answer (target for training) ─────────────────")
answer = sample["label"][0] if isinstance(sample["label"], list) else sample["label"]
label_ids = processor.tokenizer(str(answer), return_tensors="pt").input_ids
print(f"  Answer: '{answer}'")
print(f"  Answer tokens: {processor.tokenizer.convert_ids_to_tokens(label_ids[0])}")
print(f"  Answer token IDs: {label_ids[0].tolist()}")

### Answer these before moving on: [SOLUTION]

**1.** If `pixel_values.shape = (1, 3, 768, 768)` then `(768/16) * (768/16) = 48 * 48 = 2304` patch tokens.
   (Florence-2 uses 768x768 by default — significantly more tokens than ViT-B at 224x224.)

---

**2.** Lab 3's tokenizer takes text → returns only `input_ids` (integer token IDs).
   Florence-2's processor takes BOTH text AND image → returns `input_ids` (text tokens) AND `pixel_values` (normalized image tensor).
   The processor is a multimodal coordinator: it calls the text tokenizer AND the image feature extractor, then bundles their outputs.
   The model itself fuses them internally (image patches are embedded and prepended/interleaved with text tokens).

---

**3.** Florence-2 was trained with different task tokens signaling different heads/behaviors:
   - `<DocVQA>`: answer a question about document/chart content → short, specific answer
   - `<OCR>`: transcribe all text in the image → longer, verbatim output
   - `<CAPTION>`: describe the image in natural language → full sentence description
   The task token conditions the decoder's generation style. It acts like a "mode switch" the model learned during multi-task pretraining.

---

**4.** The modules NOT being updated (frozen) are the vision encoder (typically named `vision_tower` or `image_model`) and the token embedding layer.
   With LoRA configured for `["q_proj", "v_proj"]`, only those attention matrices (and the LoRA A/B factors) are trainable.
   Everything else — feed-forward layers, layer norms, vision encoder — remains frozen.

### 4.3 LoRA Configuration

In [ ]:
print("Before LoRA:")
total_before = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_before/1e6:.1f}M")
print(f"  All trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M")

lora_config = LoraConfig(
    r=8,
    # SOLUTION: r is the rank of the LoRA matrices A and B.
    # Higher r = more expressivity (larger A and B) = more trainable params = more memory.
    # r=8 means each frozen weight W (d×k) gets an update B·A where A is (8×k), B is (d×8).

    lora_alpha=16,
    # SOLUTION: lora_alpha scales the LoRA update. Effective scale = lora_alpha / r = 16/8 = 2.
    # Setting alpha=2*r is a common heuristic that keeps the update at unit scale regardless of r.
    # Larger alpha = stronger LoRA signal (can overpower the frozen weights).

    target_modules=["q_proj", "v_proj"],
    # SOLUTION: We target q_proj and v_proj (query and value projections in attention).
    # k_proj is often skipped because the key matrix determines WHAT to attend to (structural),
    # while q and v determine HOW to query and WHAT to extract — more task-specific.
    # Adding more modules increases params and expressivity but also VRAM.

    lora_dropout=0.05,
    # SOLUTION: Dropout applied to the LoRA output, preventing overfitting on small datasets.
    # On 150 examples, even a small adapter can overfit without regularization.

    bias="none",
    # SOLUTION: We do not train bias terms because bias vectors are small (d values vs d*k for weights)
    # and adding them does not significantly improve results. Keeping them frozen simplifies merging.

    task_type="SEQ_2_SEQ_LM"
    # SOLUTION: Florence-2 is an encoder-decoder (seq2seq) model, not a decoder-only causal LM.
    # This tells PEFT to apply LoRA to the decoder's attention layers correctly.
    # CAUSAL_LM would be for GPT-style models (decoder only, no encoder).
)

model = get_peft_model(model, lora_config)

print("\nAfter LoRA:")
model.print_trainable_parameters()
print()
print("Notice: only a small fraction of parameters are trainable.")
print("The vision encoder (DaViT) is completely frozen.")
print("We are only teaching the decoder to interpret projected visual tokens for THIS task.")

### 4.4 Training Loop [SOLUTION — gradient accumulation implemented]

In [ ]:
BATCH_SIZE = 16
ACCUMULATION_STEPS = 4
NUM_TRAIN = 1000

print(f"Loading ChartQA training data ({NUM_TRAIN} examples)...")
train_data = load_dataset("ahmed-masry/chartqa", split="train").select(range(NUM_TRAIN))
print(f"✅ Loaded {len(train_data)} training examples")


class ChartQADataset(Dataset):
    def __init__(self, data, proc, task_prefix=TASK_PREFIX):
        self.data = data
        self.processor = proc
        self.task_prefix = task_prefix

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        image = open_image(example["image"]).convert("RGB")
        question = self.task_prefix + example["query"]
        answer = example["label"][0] if isinstance(example["label"], list) else example["label"]
        return image, question, str(answer)


def collate_fn(batch):
    images, questions, answers = zip(*batch)

    encoding = processor(
        images=list(images),
        text=list(questions),
        return_tensors="pt",
        padding=True,
    )

    label_encoding = processor.tokenizer(
        list(answers),
        return_tensors="pt",
        padding="max_length",
        max_length=64,
        truncation=True
    )
    labels = label_encoding.input_ids.clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    encoding["labels"] = labels
    return encoding


train_dataset = ChartQADataset(train_data, processor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

print(f"Training batches per epoch: {len(train_loader)}")
print(f"Effective batch size with gradient accumulation: {BATCH_SIZE * ACCUMULATION_STEPS} (batch_size={BATCH_SIZE} × acc_steps={ACCUMULATION_STEPS})")

In [ ]:
# Training loop with gradient accumulation
# On T4 Colab: ~12-18 minutes for 3 epochs.

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)

NUM_EPOCHS = 3
train_losses = []
model.train()

print(f"Starting training: {NUM_EPOCHS} epochs, {len(train_loader)} steps/epoch")
print(f"Gradient accumulation: every {ACCUMULATION_STEPS} steps (effective batch size = {BATCH_SIZE * ACCUMULATION_STEPS})")
print()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        # SOLUTION: Gradient accumulation.
        scaled_loss = loss / ACCUMULATION_STEPS
        scaled_loss.backward()

        is_last_step = (step == len(train_loader) - 1)
        if (step + 1) % ACCUMULATION_STEPS == 0 or is_last_step:
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item()

        if (step + 1) % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"\nEpoch {epoch+1} avg loss: {avg_loss:.4f}\n")

# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Training Loss — Florence-2 LoRA Fine-tuning on ChartQA")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.5 Before / After Comparison


In [ ]:
def relaxed_accuracy(pred: str, gts: list) -> bool:
    """ChartQA's standard metric: exact match OR within 5% for numeric answers."""
    pred = str(pred).strip().lower()
    for gt in gts:
        gt = str(gt).strip().lower()
        if pred == gt:
            return True
        # Numeric tolerance check
        try:
            p_val = float(pred.replace('%', '').replace(',', ''))
            g_val = float(gt.replace('%', '').replace(',', ''))
            if abs(p_val - g_val) / (abs(g_val) + 1e-9) <= 0.05:
                return True
        except ValueError:
            pass
    return False


print("── Before / After Comparison ──────────────────────────────────\n")
model.eval()

before_correct, after_correct = 0, 0
print(f"{'Ex':<4} {'Question':<38} {'GT':<12} {'Zero-Shot':<18} {'Fine-Tuned':<18}")
print("-" * 95)

for i, ex in enumerate(eval_examples):
    image = open_image(ex["image"]).convert("RGB")
    question = ex["query"]
    gts = ex["label"] if isinstance(ex["label"], list) else [ex["label"]]

    before_pred = zero_shot_results[i]["pred"]
    after_pred = run_inference(model, processor, image, question)

    before_ok = relaxed_accuracy(str(before_pred), gts)
    after_ok = relaxed_accuracy(str(after_pred), gts)

    before_correct += before_ok
    after_correct += after_ok

    status_b = "OK" if before_ok else "XX"
    status_a = "OK" if after_ok else "XX"

    print(f"{i+1:<4} {question[:36]:<38} {str(gts[0]):<12} "
          f"[{status_b}] {str(before_pred)[:14]:<16} [{status_a}] {str(after_pred)[:14]:<16}")

print("-" * 95)
print(f"{'Acc':<4} {'':<38} {'':<12} {before_correct}/{len(eval_examples):<16} {after_correct}/{len(eval_examples):<16}")
print()
print("Reflection: What changed? What did NOT change? Write 2 sentences here.")
print("   YOUR ANSWER:")

---
## Section 5: Failure Mode Hunt

In [ ]:
model.eval()
test_examples = [chartqa[i] for i in range(10, 30)]

print("Running inference on 20 test examples to find failure cases...\n")
failures = []
for i, ex in enumerate(test_examples):
    image = open_image(ex["image"]).convert("RGB")
    question = ex["query"]
    gts = ex["label"] if isinstance(ex["label"], list) else [ex["label"]]
    pred = run_inference(model, processor, image, question)

    if not relaxed_accuracy(str(pred), gts):
        failures.append({
            "idx": i + 10,
            "image": image,
            "question": question,
            "gts": gts,
            "pred": pred
        })

print(f"Found {len(failures)} failure cases out of 20 examples.\n")

for j, fail in enumerate(failures[:3]):
    print(f"Failure {j+1} (test idx {fail['idx']}):")
    print(f"  Q: {fail['question']}")
    print(f"  GT: {fail['gts']}")
    print(f"  Pred: {fail['pred']}")
    print()

if failures:
    n_show = min(3, len(failures))
    fig, axes = plt.subplots(n_show, 1, figsize=(10, 5 * n_show))
    if n_show == 1:
        axes = [axes]
    for j in range(n_show):
        axes[j].imshow(failures[j]["image"])
        axes[j].set_title(
            f"Q: {failures[j]['question']}\n"
            f"GT: {failures[j]['gts'][0]}  |  Pred: {failures[j]['pred']}",
            fontsize=10, loc='left'
        )
        axes[j].axis('off')
    plt.suptitle("Failure Cases", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 📝 Failure Analysis — Fill in this table for your most interesting failure [SOLUTION]

| Property | Example Answer |
|---|---|
| What type of chart is it? (bar / line / pie / other) | Bar chart |
| What is the model's prediction? | 45 |
| What is the correct answer? | 42.3 |
| What reasoning step does the model appear to skip? | Reading the exact value from the y-axis scale rather than approximating the bar height |
| Is this a **vision failure**, **language failure**, or **reasoning failure**? | Vision failure — the model correctly identifies the right bar but misreads its precise height against the y-axis grid |
| What training data change might fix it? | More training examples where the correct answer requires reading fine-grained y-axis values (not just round numbers); or higher image resolution so the axis markings are clearer to the patch encoder |